In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_DS08a-009.h5
/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_DS08c-008.h5
/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_DS05.h5
/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_DS08d-010.h5
/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/Run_to_Failure_Simulation_Under_Real_Flight_Conditions_Dataset.pdf
/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_DS03-012.h5
/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_DS01-005.h5
/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_Example_data_loading_and_exploration.ipynb
/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_DS04.h5
/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_DS02-006.h5
/kaggle/input/datasets/bishals098/nasa-cmapss-2-

In [3]:
# ============================================================
# STEP 3 — Extract everything & build the dataframe
# ============================================================

FILE_PATH = '/kaggle/input/datasets/bishals098/nasa-cmapss-2-engine-degradation/N-CMAPSS_DS01-005.h5'

with h5py.File(FILE_PATH, 'r') as f:
    # Get column names from the _var keys
    a_cols  = [x.decode() for x in f['A_var'][:]]
    w_cols  = [x.decode() for x in f['W_var'][:]]
    xs_cols = [x.decode() for x in f['X_s_var'][:]]

    A  = pd.DataFrame(f['A_dev'][:],   columns=a_cols)
    W  = pd.DataFrame(f['W_dev'][:],   columns=w_cols)
    Xs = pd.DataFrame(f['X_s_dev'][:], columns=xs_cols)
    Y  = pd.DataFrame(f['Y_dev'][:],   columns=['RUL'])

df = pd.concat([A, W, Xs, Y], axis=1)

# Clean up unit & cycle columns
df['unit']  = df['unit'].astype(int)
df['cycle'] = df['cycle'].astype(int)
df['RUL']   = df['RUL'].astype(int)

print(f"✅ {df['unit'].nunique()} engines, {len(df):,} rows")
print(f"   Columns: {list(df.columns)}")
print(df[['unit','cycle','RUL']].head(10))

✅ 6 engines, 4,906,636 rows
   Columns: ['unit', 'cycle', 'Fc', 'hs', 'alt', 'Mach', 'TRA', 'T2', 'T24', 'T30', 'T48', 'T50', 'P15', 'P2', 'P21', 'P24', 'Ps30', 'P40', 'P50', 'Nf', 'Nc', 'Wf', 'RUL']
   unit  cycle  RUL
0     1      1   99
1     1      1   99
2     1      1   99
3     1      1   99
4     1      1   99
5     1      1   99
6     1      1   99
7     1      1   99
8     1      1   99
9     1      1   99


In [4]:
# ============================================================
# STEP 4 — Add alert levels
# ============================================================

def alert_level(rul):
    if rul < 10:   return 'CRITICAL'
    elif rul < 20: return 'WARNING'
    elif rul < 30: return 'WATCH'
    else:          return 'OK'

df['alert'] = df['RUL'].apply(alert_level)

# Quick summary
print(df['alert'].value_counts())
print(f"\nMin RUL: {df['RUL'].min()} | Max RUL: {df['RUL'].max()}")

alert
OK          3253173
WATCH        576912
CRITICAL     544677
WARNING      531874
Name: count, dtype: int64

Min RUL: 0 | Max RUL: 99


In [5]:
# ============================================================
# STEP 5 — Export CSV
# ============================================================

OUTPUT_PATH = '/kaggle/working/engine_monitoring.csv'
df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Done! File size: {os.path.getsize(OUTPUT_PATH)/1024/1024:.1f} MB")
print(f"   Ready to download and import into Power BI")

✅ Done! File size: 1528.1 MB
   Ready to download and import into Power BI


In [6]:
# ============================================================
# Reduce file size — sample 1 row every 10 cycles per engine
# ============================================================

df_small = df[df['cycle'] % 10 == 0].copy()

# Also keep the last 50 cycles of each engine (most important for RUL alerts)
df_last = df.groupby('unit').apply(lambda x: x.nlargest(50, 'cycle')).reset_index(drop=True)

# Combine both and remove duplicates
df_final = pd.concat([df_small, df_last]).drop_duplicates().reset_index(drop=True)

# Export the smaller version
OUTPUT_PATH = '/kaggle/working/engine_monitoring.csv'
df_final.to_csv(OUTPUT_PATH, index=False)

size_mb = os.path.getsize(OUTPUT_PATH)/1024/1024
print(f"✅ Reduced: {len(df):,} → {len(df_final):,} rows")
print(f"   File size: {size_mb:.1f} MB")

/tmp/ipykernel_55/2955900850.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_last = df.groupby('unit').apply(lambda x: x.nlargest(50, 'cycle')).reset_index(drop=True)


✅ Reduced: 4,906,636 → 488,198 rows
   File size: 152.1 MB
